# **Object Detection and Avoid Collision using AI in Autonomous Driving**

Dataset: https://www.kaggle.com/datasets/aayusmaanjain/bdd100k-for-self-driving-cars/data

required libraries

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from glob import glob
import matplotlib.pyplot as plt




from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications.vgg16 import decode_predictions

In [ ]:
# Set up TensorFlow to use the GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)

capture data to train a model for object dection on road

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!kaggle datasets download -d aayusmaanjain/bdd100k-for-self-driving-cars
!unzip /content/bdd100k-for-self-driving-cars.zip

Data Collection and Preprocessing

Initialising constants

In [ ]:
labels = [
    "bike",
    "bus",
    "car",
    "motor",
    "person",
    "rider",
    "traffic light",
    "traffic sign",
    "train",
    "truck"
]
train_path='/content/Data/train'
val_path='/content/Data/val'
model_path='/content/Data/best.pt'
inference_path='/content/Data/inference_vid.mp4'
steering_wheel_path='/content/Data/steering_wheel_image.jpg'

In [ ]:
# Get paths of all train images
train_img = glob(f'{train_path}/*.jpg')

# Filter train samples with valid labels
valid_train_samples = []
for img_path in train_img:
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    label_file = os.path.join(train_path, f'{img_id}.txt')
    if os.path.exists(label_file):
        valid_train_samples.append((img_path, label_file))

In [ ]:
# Check if directories contain image files
print("Contents of train directory:")
print(glob(f'{train_path}/*.jpg'))

print("\nContents of val directory:")
print(glob(f'{val_path}/*.jpg'))

Data Analysis

In [ ]:
train_img = glob(f'{train_path}/*.jpg')
val_img = glob(f'{val_path}/*.jpg')
n_samples = 15
train_sample = np.random.choice(len(valid_train_samples), size=n_samples, replace=False)
val_sample = np.random.choice(val_img, size=n_samples)

In [ ]:
# Debug prints
print("Contents of train_img:")
print(train_img)
print("\nContents of train_sample:")
print(train_sample)

In [ ]:
# Process and display selected train samples
IMG_HEIGHT = 720
IMG_WIDTH = 1280

for i in train_sample:
    img_path, label_file = valid_train_samples[i]
    _, ax = plt.subplots(figsize=(16, 6))

    print(f"Loading image: {img_path}")
    img = cv2.imread(img_path)
    if img is None:
        print(f"Error loading image: {img_path}")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"Processing labels for image: {img_path}")

    if not os.path.exists(label_file):
        print(f"Label file '{label_file}' not found. Skipping image.")
    else:
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for label in lines:
            splits = label.split()
            category = labels[int(splits[0])]
            x_center = float(splits[1]) * IMG_WIDTH
            y_center = float(splits[2]) * IMG_HEIGHT
            width = float(splits[3]) * IMG_WIDTH
            height = float(splits[4]) * IMG_HEIGHT

            pt1_x = x_center - width/2
            pt1_y = y_center - height/2
            pt2_x = x_center + width/2
            pt2_y = y_center + height/2

            pt1 = (int(pt1_x), int(pt1_y))
            pt2 = (int(pt2_x), int(pt2_y))

            img = cv2.rectangle(img, pt1=pt1, pt2=pt2, color=(0, 255, 0), thickness=2)
            img = cv2.putText(img, category, org=pt1,
                              color=(0, 255, 0), fontFace=cv2.FONT_HERSHEY_COMPLEX,
                              fontScale=1, thickness=2)

    ax.imshow(img)
    ax.axis('off')
    plt.show()


Explore the gathered data

In [ ]:
import yaml
import torch
import pandas as pd
from pathlib import Path

In [ ]:
! git clone https://github.com/ultralytics/yolov5.git

In [ ]:
# Install dependencies
!pip install -qr requirements.txt

# path to save the YOLOv5 training script
yolov5_path = '/content/yolov5'

# Clone YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git $yolov5_path
%cd $yolov5_path

# Define training settings in a YAML file
data = dict(
    train = train_path,
    val = val_path,
    nc = 10,  # number of classes
    names = ['bike', 'bus', 'car', 'motor', 'person', 'rider', 'traffic light', 'traffic sign', 'train', 'truck']  # class names
)

# Write data YAML file
with open('data.yaml', 'w') as f:
    yaml.dump(data, f)

# Start training
!python train.py --img 640 --batch 16 --epochs 10 --data data.yaml --cfg models/yolov5s.yaml --weights '' --noautoanchor --nosave

In [ ]:
# Define paths
model_save_dir = '/content/trained_models'
video_path = inference_path

# Check if the model weights exist
if not Path(model_save_dir).exists():
    print("Model weights not found. Please train the model first.")
    exit()

# Load the YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'custom', path_or_model=model_save_dir)

# Load the video
cap = cv2.VideoCapture(video_path)

# Define output video writer
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter('/content/inference_output.mp4', cv2.VideoWriter_fourcc(*'MP4V'), fps, (frame_width, frame_height))

# Process each frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform inference on the frame
    results = model(frame)

    # Visualize detections on the frame
    frame = results.render()[0]

    # Write the frame to the output video
    out.write(frame)

    # Display the frame
    cv2.imshow('Frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and video writer objects
cap.release()
out.release()
cv2.destroyAllWindows()